# Stage 1 — Cohort Selection

Select MIMIC-IV patients with discharge notes and ICD-10 ground truth.

**Full mode:** 15 patients, ≥2 admissions each  
**Test mode:** 1 patient (`TEST_MODE = True` in `pipeline_config.py`)

**Output:** `data/cohort/cohort.pkl` (or `data/test/cohort/` in test mode)

**Next:** Run `stage_02_information_extraction.ipynb`


## Setup

In [ ]:
import sys
from pathlib import Path

# Allow imports from repo root
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "cohort_selection.py").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from cohort_selection import load_mimic_cohort, save_cohort, load_cohort_index
from pipeline_config import (
    N_PATIENTS,
    MIN_ADMISSIONS_PER_PATIENT,
    MIN_NOTE_CHARS,
    RANDOM_SEED,
    COHORT_PICKLE,
    COHORT_INDEX_JSON,
    PHYSIONET_ROOT,
)

pd.set_option("display.max_colwidth", 120)
print(f"MIMIC root: {PHYSIONET_ROOT}")
print(f"Target patients: {N_PATIENTS} | seed: {RANDOM_SEED}")

from pipeline_config import print_pipeline_banner
print_pipeline_banner()


## Load & Sample Cohort

In [ ]:
cohort_df = load_mimic_cohort(
    n_patients=N_PATIENTS,
    min_admissions=MIN_ADMISSIONS_PER_PATIENT,
    min_note_chars=MIN_NOTE_CHARS,
    seed=RANDOM_SEED,
)

print(f"Patients: {cohort_df['subject_id'].nunique()}")
print(f"Admissions: {len(cohort_df)}")
cohort_df[[
    "patient_id", "admission_id", "gender", "anchor_age",
    "admittime", "admission_type", "primary_icd_code", "primary_dx_title", "text_len"
]].head(10)


## Patient Summary

In [ ]:
cohort_df.groupby("patient_id").agg(
    admissions=("admission_id", "count"),
    admission_types=("admission_type", lambda x: ", ".join(sorted(set(x.dropna())))),
    primary_dx=("primary_dx_title", lambda x: " | ".join(x.dropna().unique()[:3])),
).reset_index()


## Preview One Note

In [ ]:
sample = cohort_df.iloc[0]
print(f"subject_id={sample['subject_id']} hadm_id={sample['hadm_id']}")
print(f"Primary ICD-10: {sample['primary_icd_code']} — {sample['primary_dx_title']}")
print(f"All codes ({sample['n_diagnoses']}): {sample['ground_truth_icd10'][:5]}...")
print("-" * 60)
print(sample["clinical_note"][:1200], "...")


## Save Cohort (for Stage 2+)

In [ ]:
saved_path = save_cohort(cohort_df, seed=RANDOM_SEED)
index = load_cohort_index()

print(f"Saved cohort → {saved_path}")
print(f"Index       → {COHORT_INDEX_JSON}")
print(f"Patients    → {index['n_patients']}")
print(f"Admissions  → {index['n_admissions']}")
print("Subject IDs:", index["subject_ids"])
